# LeetCode #190 — Reverse Bits (Đảo thứ tự 32 bit)

**Vị trí 44 trong danh sách Easy. Python 3 — bài nhập môn thao tác bit, học từ con số 0.**  
Đề gốc: https://leetcode.com/problems/reverse-bits/

**Khung nộp LeetCode (Python 3):**
```python
class Solution:
    def reverseBits(self, n: int) -> int:
        ...
```

**Mục tiêu học sau notebook:**
1. Hiểu “bit” là gì, hệ nhị phân khác hệ thập phân thế nào và **vì sao phải đủ 32 bit**.
2. Dùng `bin`, `format`, `int(s,2)`, slicing `[::-1]`, `&`, `|`, `<<`, `>>` bằng các ví dụ nhỏ chạy được.
3. Tự giải bài đầu tiên bằng **chuỗi**, rồi tự suy ra cách ghép các bit bằng toán tử bit.
4. Trace chi tiết từng vòng lặp: bit cuối lấy ra, n bị dịch, kết quả tích lũy thay đổi.
5. Hiểu vì sao vòng lặp phải chạy **đúng 32 lần**, kể cả khi n trở thành 0; biết cả hai cách tiếp cận đều đúng.
6. Tự viết `class Solution`, làm test mẫu/biên, giải thích Big-O, phát hiện lỗi và tự luyện 6 bài từ dễ đến khó.

**Cách học:** đọc Markdown trước, tự dự đoán output cell Python rồi mới chạy; không cần thư viện ngoài.

## 1. Bài toán muốn ta “đảo” cái gì?

Trong máy tính, số nguyên có thể được biểu diễn bằng chuỗi **bit**: từng ký tự `0` hoặc `1`. Đề cho số nguyên `n`, yêu cầu đảo thứ tự **32 bit của cách biểu diễn n**, rồi trả về **giá trị số nguyên** ứng với dãy bit sau khi đảo.

Ví dụ chính của LeetCode:

```text
n = 43261596

32 bit ban đầu:  00000010100101000001111010011100
32 bit sau đảo: 00111001011110000010100101000000

kết quả số nguyên = 964176192
```

**Cực kỳ quan trọng:** ta đảo **bit**, không đảo chữ số thập phân của `43261596`. Và phải đảo **đúng 32 bit**; các bit `0` đứng đầu cũng là một phần của dữ liệu và sẽ đi về phía cuối.

Theo mô tả LeetCode được đối chiếu cho đợt này, giới hạn công bố là `0 <= n <= 2^31 - 2` và `n` chẵn; các ví dụ n lẻ hoặc lớn hơn phạm vi đó trong notebook chỉ dùng để **khảo sát thuật toán 32 bit tổng quát**.

Bài hiện tại nói về mẫu **32-bit**; ở phạm vi Python học ở đây, ta thực hành với `n` không âm hợp lệ, không cần phân tích biểu diễn âm hai bù.

In [1]:
n_example190 = 43261596
before190 = format(n_example190, '032b')
after190 = before190[::-1]
print('n =', n_example190)
print('Trước:',before190, 'độ dài=',len(before190))
print('Sau:  ',after190, 'độ dài=',len(after190))
print('Số tương ứng:',int(after190,2))
assert int(after190,2)==964176192

n = 43261596
Trước: 00000010100101000001111010011100 độ dài= 32
Sau:   00111001011110000010100101000000 độ dài= 32
Số tương ứng: 964176192


## 2. Từ số thập phân sang nhị phân: chỉ có hai chữ số 0, 1

**Hệ thập phân** quen thuộc dùng các chữ số 0 đến 9: ví dụ `13 = 1 × 10 + 3`.

**Hệ nhị phân (base 2)** chỉ dùng `0` và `1`, và các “hàng” có trọng số lần lượt là **1, 2, 4, 8, 16...** (lũy thừa của 2).

Ví dụ:

```text
13 (thập phân) = 8 + 4 + 1
13 (nhị phân)  = 1 1 0 1
                 8 4 2 1 (trọng số)
```

Đọc `1101` từ trái sang: có 8, có 4, không có 2, có 1 → tổng 13. Việc đảo các bit `1101` thành `1011` sẽ tạo số 11 **nếu chỉ xét 4 bit**. Nhưng bài #190 yêu cầu **32 bit**, nên kết quả khi đảo 13 sẽ **không phải 11**.

In [2]:
for number in [0,1,2,3,5,13,16]:
    print(f'{number:>2} (thập phân) = {number:b} (nhị phân)')
print('1101 ở hệ 2 =',8+4+0+1)

 0 (thập phân) = 0 (nhị phân)
 1 (thập phân) = 1 (nhị phân)
 2 (thập phân) = 10 (nhị phân)
 3 (thập phân) = 11 (nhị phân)
 5 (thập phân) = 101 (nhị phân)
13 (thập phân) = 1101 (nhị phân)
16 (thập phân) = 10000 (nhị phân)
1101 ở hệ 2 = 13


## 3. Hiểu `bin(n)` và tiền tố `0b`

Python có hàm `bin(n)` cho ta **chuỗi** biểu diễn số nhị phân với phần mở đầu `0b`. Ví dụ:

- `bin(13)` → `'0b1101'`.
- `bin(0)` → `'0b0'`.

`0b` là cách Python thông báo “đây là dữ liệu hệ 2”, **không phải hai bit cần đảo**. Hàm `bin` còn không tự thêm đủ các số 0 ở phía trái để thành 32 bit.

Vì vậy đảo thẳng `bin(n)[::-1]` là sai ý nghĩa: vừa đảo nhầm `'0b'`, vừa bỏ những số 0 dẫn đầu. Ta sẽ học `format(n, '032b')` để có **đúng 32 ký tự 0/1**.

In [3]:
for number in [0,5,13]:
    print('n =',number,'| bin(n) =',bin(number),'| kiểu =',type(bin(number)).__name__)

n = 0 | bin(n) = 0b0 | kiểu = str
n = 5 | bin(n) = 0b101 | kiểu = str
n = 13 | bin(n) = 0b1101 | kiểu = str


## 4. `format(n, '032b')` nghĩa là gì? Tách từng ký tự cú pháp

`format(n, '032b')` tạo một **chuỗi nhị phân dài ít nhất 32 ký tự**:

- `b` = binary (nhị phân).
- `32` = độ rộng tối thiểu 32 ký tự.
- `0` đứng trước `32` = nếu thiếu độ dài, **đệm bằng số 0 phía trái**.

Ví dụ `n=5`:

```text
5         → '101'
5 (8 bit) → '00000101'
5 (32 bit)→ '00000000000000000000000000000101'
```

Trong phạm vi `n` không âm nằm trong 32 bit, độ dài sẽ **đúng 32**. Khi biểu diễn số 32 bit, những số 0 đứng trái rất quan trọng: sau phép đảo chúng chuyển thành các số 0 đứng phải, thay đổi vị trí trọng số của những bit 1.

In [4]:
number=5
for width in [4,8,16,32]:
    s=format(number,f'0{width}b')
    print(f'{width:>2} bit:',s,'len=',len(s))
assert len(format(5,'032b'))==32

 4 bit: 0101 len= 4
 8 bit: 00000101 len= 8
16 bit: 0000000000000101 len= 16
32 bit: 00000000000000000000000000000101 len= 32


## 5. Chuỗi `str`, chỉ số `index` và slicing `[::-1]`

Trong Python, một chuỗi là dãy ký tự có thứ tự: `s = '1101'`.

```text
Ký tự:  1  1  0  1
Chỉ số: 0  1  2  3
```

- `s[0]`: ký tự ở đầu.
- `s[-1]`: ký tự cuối.
- `s[::-1]`: tạo chuỗi **đảo thứ tự toàn bộ các ký tự**.

Cú pháp slicing là `s[start:stop:step]`: khi `step=-1`, Python đi **lùi**. Với `[::-1]`, bỏ trống start/stop nghĩa là dùng toàn dãy theo hướng lùi.

`[::-1]` đảo **ký tự** chứ không trực tiếp tính số nguyên. Ta phải kết hợp `int(...,2)` sau đó.

In [5]:
s190='1101'
print('s =',s190)
print('s[0]=',s190[0],'| s[-1]=',s190[-1])
print('s[::-1]=',s190[::-1])
assert s190[::-1]=='1011'

s = 1101
s[0]= 1 | s[-1]= 1
s[::-1]= 1011


## 6. `int(binary_string, 2)` đổi chuỗi nhị phân thành số nguyên

`int('1011', 2)` nói với Python: “hãy đọc chuỗi `1011` trong **hệ cơ số 2**”.

Tính tay: `1011₂ = 1×8 + 0×4 + 1×2 + 1×1 = 11`.

- `int('1011',2)` → 11.
- `int('1101',2)` → 13.
- `int('1101')` (không có `,2`) → 1101 **thập phân**; đây là lỗi dễ mắc.

Sau khi biết ba công cụ `format` + `[::-1]` + `int(...,2)`, ta có ngay một cách giải rất trực quan, dễ kiểm chứng.

In [6]:
print('1011 hệ 2:',int('1011',2))
print('1011 mặc định hệ 10:',int('1011'))
print('0010 hệ 2:',int('0010',2))
assert int('1011',2)==11

1011 hệ 2: 11
1011 mặc định hệ 10: 1011
0010 hệ 2: 2


## 7. Cách 1 — Brute Force trực giác bằng chuỗi

Ta có thể giải bằng ba bước:

1. Biểu diễn `n` dưới dạng **chuỗi 32 bit** với `format(n,'032b')`.
2. Dùng `[::-1]` đảo chuỗi.
3. Dùng `int(reversed_bits,2)` đổi ngược về số nguyên để trả kết quả.

```python
bits = format(n, '032b')
reversed_bits = bits[::-1]
return int(reversed_bits, 2)
```

Đây là **lời giải đúng**, không phải “thuật toán sai cần bỏ”. Nó cũng khá dễ đọc và phù hợp với 32 bit cố định; nhưng ta sẽ học cách **không phải tạo chuỗi**, từ đó hiểu bit manipulation.

In [7]:
def reverse_bits_string(n: int) -> int:
    bits = format(n,'032b')
    reversed_bits = bits[::-1]
    return int(reversed_bits,2)

for n, expected in [(43261596,964176192),(0,0),(1,2147483648),(2,1073741824)]:
    print('n=',n,'->',reverse_bits_string(n))
    assert reverse_bits_string(n)==expected

n= 43261596 -> 964176192
n= 0 -> 0
n= 1 -> 2147483648
n= 2 -> 1073741824


## 8. Mô phỏng cách chuỗi với số nhỏ (chỉ 8 bit)

Trước khi xem 32 bit, hãy tưởng tượng bài **thu nhỏ thành 8 bit**, chỉ để học:

```text
n = 13
Trước: 00001101
Sau:   10110000
Giá trị = 128 + 32 + 16 = 176
```

Nếu ta đảo chuỗi `'1101'` **không thêm 0**, kết quả chỉ `'1011'` = 11, **không bằng** kết quả của phép đảo 8 bit (176). Với bài #190, ta phải áp dụng chính xác tư duy này cho **32 bit**, không phải 8 bit.

Hãy so sánh 4, 8 và 32 bit của cùng n=13.

In [8]:
def reverse_bits_width_string(n,width):
    assert 0 <= n < (1<<width)
    return int(format(n,f'0{width}b')[::-1],2)

for width in [4,8,32]:
    before=format(13,f'0{width}b')
    print(f'{width:>2} bit: {before} -> {before[::-1]} -> {reverse_bits_width_string(13,width)}')

 4 bit: 1101 -> 1011 -> 11
 8 bit: 00001101 -> 10110000 -> 176
32 bit: 00000000000000000000000000001101 -> 10110000000000000000000000000000 -> 2952790016


## 9. Vì sao muốn học tiếp cách không cần chuỗi?

Cách chuỗi vừa rồi rất phù hợp cho người mới và **đã giải đúng đề**. Nhưng nó tạo ra:

- Một chuỗi 32 ký tự.
- Một chuỗi mới sau khi đảo.
- Một bước parse chuỗi về số.

Ta có thể mô phỏng đúng các thao tác đó bằng **toán tử bit** trên số nguyên: lấy bit cuối ra, dịch kết quả sang trái, rồi gắn bit đó vào cuối. Khi bài có kích thước bit lớn thay vì cố định 32, cách làm này tiết kiệm bộ nhớ trung gian hơn (O(1) không tính kích thước số nguyên theo mô hình máy).

Trước khi ghép thành thuật toán, ta sẽ học **từng toán tử mới bằng số rất nhỏ**: `&`, `>>`, `<<`, `|`.

## 10. Bitwise `&`: lấy bit cuối bằng `n & 1`

Dấu `&` là **bitwise AND** (VÀ theo từng bit). Quy tắc từng vị trí:

| bit a | bit b | a & b |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Đặc biệt: số `1` trong nhị phân chỉ có **bit cuối** bằng 1. Khi tính `n & 1`, tất cả vị trí khác bị xóa, nên kết quả là **bit phải cùng** của n.

Ví dụ `13 = 1101₂` → `13 & 1 = 1`. `12 = 1100₂` → `12 & 1 = 0`.

**Ý nghĩa:** `last_bit = n & 1` chính là “lấy bit cuối”, tương tự `n % 2` với n không âm.

In [9]:
for n in [0,1,2,3,12,13,14]:
    print(f'n={n:>2} bits={n:04b} -> n & 1 = {n & 1}, n % 2 = {n % 2}')
    assert (n&1)==(n%2)

n= 0 bits=0000 -> n & 1 = 0, n % 2 = 0
n= 1 bits=0001 -> n & 1 = 1, n % 2 = 1
n= 2 bits=0010 -> n & 1 = 0, n % 2 = 0
n= 3 bits=0011 -> n & 1 = 1, n % 2 = 1
n=12 bits=1100 -> n & 1 = 0, n % 2 = 0
n=13 bits=1101 -> n & 1 = 1, n % 2 = 1
n=14 bits=1110 -> n & 1 = 0, n % 2 = 0


## 11. Bitwise `>>`: bỏ bit cuối bằng dịch phải

`n >> 1` là **dịch mọi bit sang phải một vị trí**. Với số nguyên không âm:

```text
13 = 1101₂
13 >> 1 → 110₂ = 6
```

Bit cuối `1` bị bỏ đi. Thao tác này có cùng kết quả với `n // 2` khi n không âm.

- `n >> 1` → bỏ **một bit cuối**.
- `n >> 2` → bỏ **hai bit cuối**.
- `n >>= 1` là viết tắt của `n = n >> 1`.

Sau khi lấy `last_bit = n & 1`, ta thường dùng `n >>= 1` để **chuyển sang bit kế tiếp**. Thứ tự rất quan trọng: lấy bit **trước khi** bỏ.

In [10]:
for n in [13,12,3,2,1,0]:
    print(f'{n:>2} ({n:04b}) >> 1 = {n>>1:>2} ({n>>1:b})')
    assert n>>1 == n//2

13 (1101) >> 1 =  6 (110)
12 (1100) >> 1 =  6 (110)
 3 (0011) >> 1 =  1 (1)
 2 (0010) >> 1 =  1 (1)
 1 (0001) >> 1 =  0 (0)
 0 (0000) >> 1 =  0 (0)


## 12. Bitwise `<<`: dành chỗ cho bit mới ở phía phải

`result << 1` là dịch các bit của result sang **trái một vị trí**, phía phải thêm bit `0`.

Ví dụ:

```text
result = 5 = 0101₂
result << 1 = 1010₂ = 10
```

Với số nguyên không âm, `x << 1` tương đương `x * 2`. Nhưng cách nghĩ theo bit hữu ích hơn: muốn **nối một bit mới vào cuối**, ta phải dịch các bit cũ sang trái để có một chỗ trống.

Khi result đang là `101`, muốn gắn bit `1` vào cuối ta cần tạo `1010`, rồi đổi bit cuối 0 thành 1 → `1011`. Toán tử `|` sẽ làm bước gắn bit.

In [11]:
for result in [0,1,5,7]:
    print(f'result={result:>2} ({result:04b}) -> <<1 = {result<<1:>2} ({result<<1:05b})')
    assert result << 1 == 2*result

result= 0 (0000) -> <<1 =  0 (00000)
result= 1 (0001) -> <<1 =  2 (00010)
result= 5 (0101) -> <<1 = 10 (01010)
result= 7 (0111) -> <<1 = 14 (01110)


## 13. Bitwise `|` (OR): gắn bit cuối đã lấy được

`|` là **bitwise OR** (HOẶC theo từng bit): chỉ cần **một** trong hai bit bằng 1 thì kết quả bằng 1.

| bit a | bit b | a \| b |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

Khi `result` vừa dịch trái, bit cuối của nó **đang bằng 0**. Vì thế `result | bit` (bit chỉ 0 hoặc 1) sẽ gắn bit mới vào vị trí trống.

Ví dụ: `1010₂ | 0001₂ = 1011₂` = 11. Nếu bit mới bằng 0, `1010₂ | 0000₂` vẫn bằng 10.

**Công thức ghép:** `result = (result << 1) | last_bit`.

Vì bit cuối đã là 0, ta cũng có thể dùng `+ last_bit`, nhưng `|` nhấn mạnh rằng đây là **thao tác ghép bit**.

In [12]:
for last_bit in [0,1]:
    result=5
    shifted=result<<1
    new=shifted|last_bit
    print(f'{result:04b} << 1 = {shifted:04b}; OR {last_bit} -> {new:04b} (= {new})')
    assert new == result*2+last_bit

0101 << 1 = 1010; OR 0 -> 1010 (= 10)
0101 << 1 = 1010; OR 1 -> 1011 (= 11)


## 14. Trước khi viết thuật toán, ghép ba bước vào một lượt

Giả sử `n = 13 = 1101₂`, `result = 0`:

1. `bit = n & 1` → bit cuối là **1**.
2. `result = (result << 1) | bit` → `0 << 1 | 1` = **1**.
3. `n >>= 1` → `13 >> 1` = **6**, có nhị phân `110₂`.

Lượt sau: lấy bit cuối của `6` là **0**, ghép vào result 1 → `10₂` (2), rồi n thành 3.

Ta **đang đọc n từ phải sang trái**, đồng thời **viết kết quả từ trái sang phải**. Khi thực hiện đủ số lượng bit, đây chính là thao tác đảo thứ tự bit.

In [13]:
n190=13
result190=0
for step in range(2):
    bit=n190 & 1
    before=result190
    result190=(result190<<1)|bit
    print(f'Lượt {step+1}: n={n190:04b}, bit={bit}, result {before:b} -> {result190:b}')
    n190 >>= 1
print('n còn lại sau 2 lượt:',n190)

Lượt 1: n=1101, bit=1, result 0 -> 1
Lượt 2: n=0110, bit=0, result 1 -> 10
n còn lại sau 2 lượt: 3


## 15. Vòng lặp `for i in range(32)` — tại sao không dừng khi `n == 0`?

`range(32)` tạo các giá trị `0, 1, ..., 31` → đúng **32 lượt**.

**Vì sao phải đúng 32?** Các bit số 0 ở bên trái của n cũng là một phần của cách biểu diễn có độ rộng 32 bit. Dù `n` đã thành 0 (không còn bit 1 nào), ta vẫn phải tiếp tục **dịch result sang trái**, tương ứng đẩy những bit 1 đã đảo về đúng vị trí cuối cùng.

Ví dụ `n=1`:

```text
32 bit trước: 00000000000000000000000000000001
32 bit sau:   10000000000000000000000000000000
               ↑ bit 1 phải được chuyển lên vị trí cao nhất
```

Giá trị sau đảo là **2³¹ = 2147483648**, không phải 1. Nếu dừng `while n > 0`, chỉ chạy một vòng nên sai!

In [14]:
for n in [0,1,2,13]:
    print('n =',n,'| bit_length =',n.bit_length(),'| vẫn phải xử lý 32 bit')
print('n=1 đảo đúng 32 bit =',1<<31)
assert reverse_bits_string(1)==1<<31

n = 0 | bit_length = 0 | vẫn phải xử lý 32 bit
n = 1 | bit_length = 1 | vẫn phải xử lý 32 bit
n = 2 | bit_length = 2 | vẫn phải xử lý 32 bit
n = 13 | bit_length = 4 | vẫn phải xử lý 32 bit
n=1 đảo đúng 32 bit = 2147483648


## 16. Lời giải chính — học từng dòng rồi mới ghép lại

```python
def reverse_bits_bitwise(n):
    result = 0
    for _ in range(32):
        bit = n & 1
        result = (result << 1) | bit
        n >>= 1
    return result
```

Giải thích:
- `result = 0`: chưa ghép bit nào.
- `_` là biến vòng lặp **không cần dùng tên**, vì ta chỉ cần chạy đủ 32 lần.
- `n & 1`: lấy bit phải nhất **chưa xử lý**.
- `(result << 1) | bit`: dịch kết quả đã ghép sang trái và gắn bit mới.
- `n >>= 1`: bỏ bit vừa đọc để lượt sau lấy bit kế tiếp.
- `return result`: trả số nguyên, không phải chuỗi.

Chúng ta không cần danh sách và cũng không cần tạo chuỗi 32 ký tự.

In [15]:
def reverse_bits_bitwise(n: int) -> int:
    result=0
    for _ in range(32):
        bit=n&1
        result=(result<<1)|bit
        n>>=1
    return result

print('Ví dụ LeetCode:', reverse_bits_bitwise(43261596))
assert reverse_bits_bitwise(43261596)==964176192

Ví dụ LeetCode: 964176192


## 17. Trace ngắn 8-bit để nhìn rõ từng lần đổi trạng thái

Với phiên bản **8 bit** minh họa n=13:

| Lượt | `n` trước | `bit=n&1` | `result` trước | `result` sau | `n` sau |
|---:|---|---:|---|---|---|
| 1 | 00001101 | 1 | 0 | 1 | 00000110 |
| 2 | 00000110 | 0 | 1 | 10 | 00000011 |
| 3 | 00000011 | 1 | 10 | 101 | 00000001 |
| 4 | 00000001 | 1 | 101 | 1011 | 00000000 |
| 5 | 00000000 | 0 | 1011 | 10110 | 00000000 |
| 6 | 00000000 | 0 | 10110 | 101100 | 00000000 |
| 7 | 00000000 | 0 | 101100 | 1011000 | 00000000 |
| 8 | 00000000 | 0 | 1011000 | 10110000 | 00000000 |

Cuối cùng `10110000₂ = 176` — đúng phép đảo 8 bit. **Chính các vòng 5–8 có n=0 vẫn làm result dịch trái**, nên không được bỏ. Sau đây ta in trace thực bằng Python.

In [16]:
def trace_reverse_bits(n,width=8):
    assert 0 <= n < (1<<width)
    result=0
    print(f'Đầu vào: n={n}, bit={n:0{width}b}')
    print('lượt | n trước | bit | result trước -> result sau | n sau')
    for i in range(width):
        old_n, old_result = n, result
        bit=n&1
        result=(result<<1)|bit
        n>>=1
        print(f'{i+1:>4} | {old_n:0{width}b} |  {bit}  | '
              f'{old_result:0{width}b} -> {result:0{width}b} | {n:0{width}b}')
    print('Kết quả:',result,'=',f'{result:0{width}b}')
    return result

assert trace_reverse_bits(13,8)==176

Đầu vào: n=13, bit=00001101
lượt | n trước | bit | result trước -> result sau | n sau
   1 | 00001101 |  1  | 00000000 -> 00000001 | 00000110
   2 | 00000110 |  0  | 00000001 -> 00000010 | 00000011
   3 | 00000011 |  1  | 00000010 -> 00000101 | 00000001
   4 | 00000001 |  1  | 00000101 -> 00001011 | 00000000
   5 | 00000000 |  0  | 00001011 -> 00010110 | 00000000
   6 | 00000000 |  0  | 00010110 -> 00101100 | 00000000
   7 | 00000000 |  0  | 00101100 -> 01011000 | 00000000
   8 | 00000000 |  0  | 01011000 -> 10110000 | 00000000
Kết quả: 176 = 10110000


## 18. `width` ở hàm trace là gì? Phân biệt minh họa và đề gốc

Hàm `trace_reverse_bits(n,width=8)` vừa rồi có tham số **width** để minh họa với 4 hay 8 bit, tránh in tới 32 dòng khó đọc.

- Khi `width=8`, ta đảo đúng **8 bit**; dùng để học cách thuật toán hoạt động.
- Khi `width=32`, ta đảo đúng **32 bit**; đây mới là yêu cầu bài LeetCode #190.
- `assert 0 <= n < (1<<width)` kiểm tra n **vừa trong độ rộng** đã chọn để mô phỏng không bị mất bit.

Các số minh họa 8-bit không phải đầu ra cần nộp cho #190. Rất nhiều lỗi đến từ việc **dùng đúng thuật toán nhưng sai độ rộng bit**.

In [17]:
print('Đảo 4 bit n=13:')
assert trace_reverse_bits(13,4)==11
print('Đảo 8 bit n=13:')
assert trace_reverse_bits(13,8)==176

Đảo 4 bit n=13:
Đầu vào: n=13, bit=1101
lượt | n trước | bit | result trước -> result sau | n sau
   1 | 1101 |  1  | 0000 -> 0001 | 0110
   2 | 0110 |  0  | 0001 -> 0010 | 0011
   3 | 0011 |  1  | 0010 -> 0101 | 0001
   4 | 0001 |  1  | 0101 -> 1011 | 0000
Kết quả: 11 = 1011
Đảo 8 bit n=13:
Đầu vào: n=13, bit=00001101
lượt | n trước | bit | result trước -> result sau | n sau
   1 | 00001101 |  1  | 00000000 -> 00000001 | 00000110
   2 | 00000110 |  0  | 00000001 -> 00000010 | 00000011
   3 | 00000011 |  1  | 00000010 -> 00000101 | 00000001
   4 | 00000001 |  1  | 00000101 -> 00001011 | 00000000
   5 | 00000000 |  0  | 00001011 -> 00010110 | 00000000
   6 | 00000000 |  0  | 00010110 -> 00101100 | 00000000
   7 | 00000000 |  0  | 00101100 -> 01011000 | 00000000
   8 | 00000000 |  0  | 01011000 -> 10110000 | 00000000
Kết quả: 176 = 10110000


## 19. Mô phỏng đủ 32 bit của ví dụ đề

Với `n=43261596`, ta sẽ in một bảng rút gọn **32 lượt**. Mỗi dòng ghi:

- Lượt i (0–31),
- bit vừa lấy từ phía phải của n,
- `n` sau khi dịch phải,
- các bit đã tích lũy vào result sau khi dịch/gắn.

Nếu bạn thấy `n` trở thành 0 mà vòng lặp vẫn chạy: **đó là đúng**, vì còn những số 0 bên trái của đầu vào cần được xử lý để result có đúng 32 vị trí.

Bảng này là **trace học tập**, còn code nộp không có `print`.

In [18]:
def trace_32_bits(n):
    result=0
    print('Lượt | bit | n sau dịch | result sau ghép (32 bit)')
    for i in range(32):
        bit=n&1
        result=(result<<1)|bit
        n>>=1
        print(f'{i+1:>4} |  {bit}  | {n:>10} | {result:032b}')
    print('Cuối cùng =',result)
    return result

assert trace_32_bits(43261596)==964176192

Lượt | bit | n sau dịch | result sau ghép (32 bit)
   1 |  0  |   21630798 | 00000000000000000000000000000000
   2 |  0  |   10815399 | 00000000000000000000000000000000
   3 |  1  |    5407699 | 00000000000000000000000000000001
   4 |  1  |    2703849 | 00000000000000000000000000000011
   5 |  1  |    1351924 | 00000000000000000000000000000111
   6 |  0  |     675962 | 00000000000000000000000000001110
   7 |  0  |     337981 | 00000000000000000000000000011100
   8 |  1  |     168990 | 00000000000000000000000000111001
   9 |  0  |      84495 | 00000000000000000000000001110010
  10 |  1  |      42247 | 00000000000000000000000011100101
  11 |  1  |      21123 | 00000000000000000000000111001011
  12 |  1  |      10561 | 00000000000000000000001110010111
  13 |  1  |       5280 | 00000000000000000000011100101111
  14 |  0  |       2640 | 00000000000000000000111001011110
  15 |  0  |       1320 | 00000000000000000001110010111100
  16 |  0  |        660 | 00000000000000000011100101111000
  17 

## 20. Một trace phản ví dụ: n=1 vì sao dừng sớm sai

`n=1` có 32 bit: `000...0001`.

- Lượt 1: lấy `bit=1`, `result=1`, `n=0`.
- Lượt 2: lấy `bit=0`, dịch result `1→10` (2).
- Lượt 3: result `10→100` (4).
- ...
- Lượt 32: result `1` theo sau 31 số 0 → `2³¹`.

Nếu dùng `while n>0` và dừng sau lượt 1, output sẽ là 1 thay vì `2³¹`. Đây là ca kiểm thử cực quan trọng dù bản mô tả LeetCode tại thời điểm tra cứu có thể đặt giới hạn input hẹp hơn (ví dụ n chẵn); ví dụ `n=1` ở đây **là thử tính tổng quát 32-bit**, không giả nhận nó nằm trong miền chính thức.

In [19]:
def reverse_bits_wrong_stop(n):
    result=0
    while n>0:
        result=(result<<1)|(n&1)
        n>>=1
    return result

print('n=1, dừng sớm:',reverse_bits_wrong_stop(1),'| đúng 32 bit:',reverse_bits_bitwise(1))
assert reverse_bits_wrong_stop(1)!=reverse_bits_bitwise(1)

n=1, dừng sớm: 1 | đúng 32 bit: 2147483648


## 21. Dừng sớm khi `n=0` có *luôn* sai? Phân biệt phiên bản có bù số lượt

Ta nói `while n>0` **một mình** là sai, vì bỏ qua các bit 0 còn lại. Nhưng nếu tự **đếm số bit đã xử lý** và tiếp tục dịch kết quả cho đủ 32 vị trí, ta có thể viết một biến thể đúng:

- Vòng while lấy các bit đáng kể của n.
- Đếm `processed` bao nhiêu bit đã đọc.
- Sau khi n=0, dịch `result <<= (32-processed)` để xử lý các bit 0 đứng đầu.

Biến thể này giúp hiểu **mấu chốt thật**: không phải “bắt buộc cú pháp `for`”, mà phải **xử lý đủ 32 vị trí**. Tuy nhiên cách `for range(32)` ngắn, dễ chứng minh và ít lỗi hơn cho người mới.

In [20]:
def reverse_bits_while_with_padding(n):
    result=0
    processed=0
    while n>0:
        result=(result<<1)|(n&1)
        n>>=1
        processed+=1
    result <<= (32-processed)
    return result

for n in [0,1,2,13,43261596]:
    assert reverse_bits_while_with_padding(n)==reverse_bits_bitwise(n)
print('PASS while có bù số lượt vẫn đúng')

PASS while có bù số lượt vẫn đúng


## 22. Code chuẩn LeetCode — đúng `class Solution` và `reverseBits`

Khi nộp bài Python 3, LeetCode tạo sẵn đối tượng `Solution()` và gọi method `reverseBits(n)`.

- `class Solution:` là khuôn mẫu yêu cầu của LeetCode.
- `def reverseBits(self, n: int) -> int:` định nghĩa method; `self` là đối tượng hiện tại.
- `: int` và `-> int` là **type hint**, giúp người đọc hiểu input/output, không phải phép tính.
- `return result` trả số nguyên; `print(result)` **không thay thế return**.

Code phía dưới **không** in debug trong method, chỉ làm đúng thuật toán đã hiểu.

In [21]:
class Solution:
    def reverseBits(self, n: int) -> int:
        result=0
        for _ in range(32):
            result=(result<<1)|(n&1)
            n>>=1
        return result

solver190=Solution()
print('LeetCode example:',solver190.reverseBits(43261596))
assert solver190.reverseBits(43261596)==964176192

LeetCode example: 964176192


## 23. Test ví dụ đề gốc và giải thích mẫu thứ hai

Ví dụ 1 đã biết: `43261596 → 964176192`.

Ví dụ 2 (theo mô tả đang tham khảo):

```text
n = 2147483644
bits trước: 01111111111111111111111111111100
bits sau:   00111111111111111111111111111110
kết quả:    1073741822
```

Không cần tự đoán con số lớn trong đầu. Hãy dùng `format(n,'032b')` để **nhìn đúng dãy bit**, rồi tự đối chiếu với kết quả method. Hai ví dụ bao phủ đầu vào có nhiều bit 1 và nhiều bit 0.

In [22]:
official_cases190=[(43261596,964176192),(2147483644,1073741822)]
for n,expected in official_cases190:
    actual=solver190.reverseBits(n)
    print('n=',n, '| input:', f'{n:032b}', '| output:', f'{actual:032b}', '| số:', actual)
    assert actual==expected
print('PASS cả hai ví dụ LeetCode')

n= 43261596 | input: 00000010100101000001111010011100 | output: 00111001011110000010100101000000 | số: 964176192
n= 2147483644 | input: 01111111111111111111111111111100 | output: 00111111111111111111111111111110 | số: 1073741822
PASS cả hai ví dụ LeetCode


## 24. Bộ test biên — 0, bit 1 ở đầu/cuối, mẫu xen kẽ

Các ca giúp kiểm tra thuật toán:

- `n=0`: 32 bit đều 0 → kết quả 0.
- `n=1`: bit cuối đi lên đầu → `2³¹` (minh họa bit tổng quát).
- `n=2`: bit `1` ở vị trí thứ hai từ phải → sau đảo đến vị trí thứ hai từ trái → `2³⁰`.
- `n=2³¹`: bit cao nhất thành bit thấp nhất → 1 (**ngoài miền n của mô tả hiện tại nếu yêu cầu n nhỏ hơn 2³¹**, chỉ là mở rộng 32 bit).
- `n` có mẫu `1010...`: đảo thành `0101...`.
- `n=2³²−1`: 32 bit đều 1, đảo vẫn vậy (**ngoài miền input chính thức hiện tại**, dùng để kiểm tra thuật toán 32-bit tổng quát).

Thử ca ngoài giới hạn đề là **bài thử tổng quát**, không thay đổi constraints của đề.

In [23]:
extended_cases190=[
    ('zero',0,0),
    ('one (mở rộng)',1,1<<31),
    ('two',2,1<<30),
    ('bit cao (mở rộng)',1<<31,1),
    ('all ones (mở rộng)',(1<<32)-1,(1<<32)-1),
    ('mẫu xen kẽ',0xAAAAAAAA,0x55555555),
]
for label,n,expected in extended_cases190:
    actual=solver190.reverseBits(n)
    assert actual==expected,(label,actual,expected)
    print(label,': PASS')

zero : PASS
one (mở rộng) : PASS
two : PASS
bit cao (mở rộng) : PASS
all ones (mở rộng) : PASS
mẫu xen kẽ : PASS


## 25. Kiểm thử so với lời giải chuỗi trên nhiều số

Ta đã có hai cách giải khá khác nhau:

- **Chuỗi**: `format(...)[::-1]`.
- **Bitwise**: lấy bit cuối và dịch/gắn 32 lần.

Nếu hai cách luôn cho cùng kết quả trên nhiều input, ta tăng độ tin tưởng rằng code không bị lỗi cú pháp hoặc xử lý thiếu bit. Dùng `random.Random(190)` với hạt giống cố định để test tái hiện được.

**Không phải chứng minh:** hai hàm có thể cùng sai nếu cùng một hiểu lầm. Ta vẫn cần lý giải invariant ở mục sau.

In [24]:
import random
rng190=random.Random(190)
for i in range(500):
    n=rng190.randrange(1<<32)
    actual=reverse_bits_bitwise(n)
    expected=reverse_bits_string(n)
    assert actual==expected,(i,n,actual,expected)
print('PASS 500 số 32 bit ngẫu nhiên: bitwise == string')

PASS 500 số 32 bit ngẫu nhiên: bitwise == string


## 26. Chứng minh vì sao thuật toán đúng — bất biến (invariant) đơn giản

**Bất biến** là một điều luôn đúng trước/sau các vòng lặp, giúp chứng minh thuật toán.

Sau `k` lượt (0 ≤ k ≤ 32):

1. Ta đã **lấy k bit bên phải nhất của n gốc**, theo thứ tự từ phải sang trái.
2. `result` chứa đúng **k bit ấy theo thứ tự đã đọc**, tức là phần đầu của dãy bit đã đảo.
3. Biến `n` hiện tại là phần **chưa xử lý** của số gốc sau khi dịch phải k lần.

**Khởi đầu:** k=0, result=0 — chưa xử lý bit nào, đúng.

**Mỗi lượt:** `n&1` đọc chính bit tiếp theo; `(result<<1)|bit` nối nó vào bên phải chuỗi kết quả đã có; `n>>=1` bỏ đúng bit vừa đọc. Nên bất biến tiếp tục đúng.

**Kết thúc:** k=32, ta đã đọc **đủ 32 bit của số gốc từ phải sang trái**, do đó result chính là giá trị của chuỗi 32 bit đảo ngược. Thuật toán không bỏ bit và không thêm bit sai.

## 27. Chứng minh bằng công thức vị trí bit (sau khi đã hiểu trực giác)

Đặt `b_i` là bit ở vị trí thứ `i` tính từ phải (bắt đầu từ i=0). Số gốc có thể viết:

\[
n = \sum_{i=0}^{31} b_i 2^i.
\]

Khi đảo thứ tự 32 bit, bit `b_i` chuyển tới vị trí `31-i`, nên số mới là:

\[
\text{reverseBits}(n) = \sum_{i=0}^{31} b_i 2^{31-i}.
\]

Ví dụ `n=2 = 2¹`: chỉ `b_1=1`. Sau đảo, nó chuyển sang vị trí `31−1=30` → `2³⁰ = 1073741824`. Trong vòng lặp, bit `b_1` được đọc ở lượt 2 và còn 30 lượt dịch trái, nên cũng kết thúc tại vị trí 30.

Bạn **không cần học thuộc công thức** để code, nhưng nó chứng minh đầy đủ tính đúng đắn của thao tác.

In [25]:
n=2
position_before=1
position_after=31-position_before
print('Bit 1 ban đầu ở vị trí:',position_before)
print('Bit 1 sau đảo ở vị trí:',position_after)
print('Giá trị=',2**position_after)
assert reverse_bits_bitwise(n)==2**position_after

Bit 1 ban đầu ở vị trí: 1
Bit 1 sau đảo ở vị trí: 30
Giá trị= 1073741824


## 28. Phân tích Time / Space đúng nghĩa

Gọi `w` là **độ rộng bit** (trong bài #190, w = 32 **cố định**).

- **Cách bitwise:** chạy đúng `w` lượt, mỗi lượt vài phép toán trên từ máy có độ rộng cố định. Thời gian `O(w)`, bộ nhớ phụ `O(1)` theo mô hình thao tác trên số có độ rộng cố định.
- **Cách chuỗi:** tạo, đảo và chuyển chuỗi `w` ký tự → `O(w)` thời gian, `O(w)` bộ nhớ trung gian.
- Vì bài này **cố định 32 bit**, cả hai có thể được mô tả là **`O(1)` thời gian** theo kích thước đầu vào (vì 32 là hằng số), còn cách chuỗi dùng **`O(1)` bộ nhớ theo kích thước bài cụ thể**. Nhưng nói `O(w)` thể hiện rõ **lý do thuật toán chạy 32 bước** và giúp hiểu khi tổng quát hóa lên w bit.

**Chú ý kỹ thuật Python:** `int` có độ dài tùy ý, nên nếu thực sự tổng quát hóa w lên vô hạn, chi phí toán số lớn phụ thuộc số bit; phân tích trên là theo mô hình thao tác bit chuẩn. Đừng nói cách chuỗi “O(32)” rồi kết luận “chậm hơn 32 lần” nếu không đo đạc.

## 29. Lỗi 1 — đảo nhầm chữ số thập phân

Code sai kiểu:

```python
int(str(n)[::-1])
```

Nó đảo **ký tự thập phân**, không đảo bit. Với `n=13`, code đó trả 31, còn đảo **32 bit** của 13 là một số khác hẳn. Đây là hai bài toán khác nhau:

- **Palindrome Number / Reverse Integer:** xử lý chữ số hệ 10.
- **Reverse Bits:** xử lý từng bit của **đúng độ rộng 32**.

Hãy kiểm tra lại đối tượng cần đảo trước khi chọn `str()` hay `format(n,'032b')`.

In [26]:
print('Sai: đảo số thập phân 13 ->',int(str(13)[::-1]))
print('Đúng: đảo 32 bit số 13 ->',reverse_bits_bitwise(13))
assert int(str(13)[::-1]) != reverse_bits_bitwise(13)

Sai: đảo số thập phân 13 -> 31
Đúng: đảo 32 bit số 13 -> 2952790016


## 30. Lỗi 2 — đảo `bin(n)` mà không đệm đủ 32 bit

`bin(5)` trả `'0b101'`, không phải 32 bit. Nếu cắt `bin(n)[2:]` để bỏ tiền tố `0b`, ta chỉ còn **3 bit**, và đảo thành `'101'` (vẫn bằng 5), trong khi đáp án 32-bit không như vậy.

**Cách sửa:** dùng `format(n,'032b')` hoặc xử lý đúng 32 lượt bằng toán tử bit.

Với `n=2`, cắt tiền tố rồi đảo chỉ ra `'01'` (=1), trong khi đảo 32 bit là `1073741824`. Đây là phản ví dụ rất dễ nhớ.

In [27]:
n=2
wrong=int(bin(n)[2:][::-1],2)
correct=reverse_bits_bitwise(n)
print('Không đệm 0:',wrong,'| đủ 32 bit:',correct)
assert wrong != correct

Không đệm 0: 1 | đủ 32 bit: 1073741824


## 31. Lỗi 3 — lấy bit sau khi đã dịch n

Code sai thứ tự:

```python
n >>= 1
bit = n & 1
```

Lệnh đó **vứt bit cuối trước**, rồi mới đọc bit tiếp theo → mất bit đầu tiên cần đảo. Với `n=1`, sau khi dịch n thành 0, bạn sẽ đọc bit=0 thay vì 1.

**Cách đúng:** `bit = n & 1` **trước**, sau đó `n >>= 1` sau khi đã ghép bit vào result. Thứ tự lệnh có ý nghĩa thuật toán, không chỉ là phong cách code.

In [28]:
n=1
n_wrong=n>>1
print('Sai: dịch n trước -> bit',n_wrong&1)
print('Đúng: lấy bit trước -> bit',n&1)
assert (n_wrong&1)!=(n&1)

Sai: dịch n trước -> bit 0
Đúng: lấy bit trước -> bit 1


## 32. Lỗi 4 — gộp bit bằng `&` thay vì `|`

Trong `result = (result<<1) | bit`, dấu `|` là **OR**, giúp gắn bit 1 vào ô trống bên phải. Nếu thay bằng `&`, một bên là số **chẵn** vừa dịch trái, một bên chỉ là 0/1: phần lớn kết quả sẽ bằng 0.

Ví dụ: `2 << 1 = 4 = 100₂`; `100₂ & 001₂ = 000₂` (sai), trong khi `100₂ | 001₂ = 101₂` (đúng).

**Nhớ chức năng:**
- `& 1` để **lấy** bit cuối.
- `<< 1` để **chừa chỗ**.
- `| bit` để **gắn** bit cuối vào chỗ trống.

In [29]:
left=2<<1
print('left:',f'{left:03b}', '| bit 1:', f'{left|1:03b}', '& bit 1:',f'{left&1:03b}')
assert (left|1)==5 and (left&1)==0

left: 100 | bit 1: 101 & bit 1: 000


## 33. Lỗi 5 — sai số lượt và sai giới hạn input

- `range(31)` chỉ chạy 31 vòng → thiếu một bit.
- `range(n)` nhầm **n là số lần lặp** → không liên quan số bit cần đảo.
- `while n > 0` không xử lý các bit 0 dẫn đầu, trừ khi có bước bù độ rộng như mục 21.
- `for i in range(32)` xử lý 32 vị trí ổn định, bất kể n có bao nhiêu bit 1.

**Giới hạn đầu vào:** Đề gốc có thể thay đổi ràng buộc trong các phiên bản. Bài này luôn nói **đảo thứ tự 32 bit**, vì thế code đã học áp dụng cho n không âm biểu diễn trong 32 bit. Các ví dụ đặc biệt như 1 hoặc 2³²−1 trong notebook được gắn nhãn minh họa tổng quát nếu ngoài giới hạn công bố.

In [30]:
print('31 lượt (sai) vs 32 lượt (đúng), n=2:')
def wrong_31_rounds(n):
    ans=0
    for _ in range(31):
        ans=(ans<<1)|(n&1)
        n>>=1
    return ans
print('31:',wrong_31_rounds(2),'| 32:',reverse_bits_bitwise(2))
assert wrong_31_rounds(2)!=reverse_bits_bitwise(2)

31 lượt (sai) vs 32 lượt (đúng), n=2:
31: 536870912 | 32: 1073741824


## 34. Mẹo nhận dạng pattern “trích bit → dịch → ghép”

Trong những bài như:

- Kiểm tra bit cuối là 0/1.
- Đếm số bit bằng 1.
- Chuyển số sang chuỗi nhị phân.
- Đảo thứ tự các bit.
- Tạo số mới từ một dãy bit đọc theo thứ tự.

Ta thường dùng cùng nhóm công cụ:

| Cần làm gì? | Python |
|---|---|
| Lấy bit cuối | `n & 1` |
| Bỏ bit cuối | `n >>= 1` |
| Mở chỗ trống bên phải của kết quả | `result <<= 1` |
| Thêm bit mới vào bên phải | `result |= bit` |

**Không học vẹt:** hiểu trạng thái của bit trước/sau mỗi thao tác; bạn sẽ tự biến đổi được pattern cho bài mới.

## 35. Biến thể 1 — đảo `w` bit (tổng quát hóa đúng cách)

Ta có thể viết hàm `reverse_bits_width(n, width)` để đảo một số độ rộng bất kỳ **được chỉ rõ**:

- `width=4`, n=13 (`1101`) → `1011` = 11.
- `width=8`, n=13 (`00001101`) → `10110000` = 176.
- `width=32`, n=13 → kết quả LeetCode (độ rộng 32).

Điều kiện `0 <= n < 2^width` đảm bảo số n không **tràn ra ngoài** w bit. Hàm này là biến thể để hiểu sâu; code nộp bài chỉ cần `width=32` cố định.

In [31]:
def reverse_bits_width(n,width):
    if width < 1 or not (0 <= n < (1<<width)):
        raise ValueError('n phải nằm trong width bit không âm')
    result=0
    for _ in range(width):
        result=(result<<1)|(n&1)
        n>>=1
    return result

assert reverse_bits_width(13,4)==11
assert reverse_bits_width(13,8)==176
assert reverse_bits_width(13,32)==reverse_bits_bitwise(13)
print('4/8/32 bit:',*[reverse_bits_width(13,w) for w in [4,8,32]])

4/8/32 bit: 11 176 2952790016


## 36. Biến thể 2 — đếm số bit 1 (popcount)

Khi đã biết `n & 1` và `n >> 1`, ta có thể đếm bao nhiêu bit **bằng 1** trong số không âm n:

1. `count=0`.
2. Lặp khi `n>0`, cộng `n&1` vào count.
3. `n >>= 1` để xét bit kế tiếp.

Bài này **khác Reverse Bits**: ta chỉ đếm, không cần tạo chuỗi mới hay đảo 32 vị trí. Với popcount, dừng khi `n=0` là hợp lý vì các bit 0 còn lại **không làm tăng count**. Điều này cho thấy điều kiện dừng phụ thuộc **mục tiêu bài toán**, không có quy tắc chung “luôn 32 lượt” cho mọi bài bit.

In [32]:
def count_ones(n):
    count=0
    while n>0:
        count+=n&1
        n>>=1
    return count

for n in [0,1,5,7,13,16]:
    print(f'{n:>2} = {n:b} có {count_ones(n)} bit 1')
assert count_ones(13)==3

 0 = 0 có 0 bit 1
 1 = 1 có 1 bit 1
 5 = 101 có 2 bit 1
 7 = 111 có 3 bit 1
13 = 1101 có 3 bit 1
16 = 10000 có 1 bit 1


## 37. Biến thể 3 — đảo 2 lần phải quay lại ban đầu

Đảo một dãy 32 bit **hai lần** sẽ trả về dãy ban đầu. Ví dụ `0011 → 1100 → 0011`.

Tính chất này là một **kiểm tra quy luật (property test)** rất mạnh:

```python
reverseBits(reverseBits(n)) == n
```

với `n` trong miền 32 bit không âm. Nó không chứng minh một mình thuật toán đúng: một hàm giữ nguyên n cũng có tính chất đảo hai lần! Nhưng kết hợp với test cụ thể và chứng minh thứ tự bit, nó phát hiện được nhiều lỗi.

Sau đây ta chạy trên nhiều số, gồm cả số ngẫu nhiên.

In [33]:
for n in [0,1,2,13,43261596,(1<<32)-1]:
    twice=reverse_bits_bitwise(reverse_bits_bitwise(n))
    print(n,'-> đảo 2 lần ->',twice)
    assert twice==n
print('PASS tính chất đảo hai lần')

0 -> đảo 2 lần -> 0
1 -> đảo 2 lần -> 1
2 -> đảo 2 lần -> 2
13 -> đảo 2 lần -> 13
43261596 -> đảo 2 lần -> 43261596
4294967295 -> đảo 2 lần -> 4294967295
PASS tính chất đảo hai lần


## 38. Follow-up: nếu hàm bị gọi rất nhiều lần thì sao?

LeetCode gợi ý: nếu `reverseBits` được gọi nhiều lần, có thể tối ưu thế nào? Một phương án là **tiền tính (precompute)** phép đảo của **8 bit** (một byte) cho tất cả `2⁸ = 256` giá trị, rồi tái sử dụng bảng đó.

Vì 32 = 4 × 8:

1. Chia n thành **4 byte**, từ byte thấp nhất đến cao nhất.
2. Tra bảng để đảo thứ tự bit **bên trong từng byte**.
3. Ghép các byte đảo **theo thứ tự ngược** vào kết quả.

Đây là mở rộng **nâng cao**, không cần học thuộc để pass #190. Lợi ích chủ yếu khi gọi hàm **nhiều lần**, vì có chi phí xây bảng 256 phần tử ban đầu. Đừng vội tối ưu trước khi hiểu lời giải 32 lượt.

## 39. Hiểu “byte” và vì sao có 256 trường hợp

Một **byte = 8 bit**, mỗi bit là 0/1. Số mẫu khác nhau của một byte là `2⁸ = 256`, từ `00000000₂` = 0 đến `11111111₂` = 255.

Chẳng hạn byte `00001101` (=13) đảo thành `10110000` (=176). Ta có thể tính trước **một bảng** `reverse_byte[13] = 176`.

Khi đảo một số 32 bit, ta tách thành:

```text
Byte 3 | Byte 2 | Byte 1 | Byte 0
  8 bit  | 8 bit  | 8 bit  | 8 bit
```

Kết quả phải đặt byte 0 (đã đảo) lên trước, rồi byte 1, byte 2, byte 3. **Đảo bit trong byte thôi là chưa đủ**: cũng cần **đảo thứ tự các byte**.

In [34]:
reverse_byte_table = [reverse_bits_width(value,8) for value in range(256)]
for value in [0,1,2,13,255]:
    print(f'byte {value:08b} -> {reverse_byte_table[value]:08b}')
assert reverse_byte_table[13]==176 and len(reverse_byte_table)==256

byte 00000000 -> 00000000
byte 00000001 -> 10000000
byte 00000010 -> 01000000
byte 00001101 -> 10110000
byte 11111111 -> 11111111


## 40. Dùng bảng đảo byte để ghép lại 32 bit

Cách ghép:

```python
result = 0
for _ in range(4):
    byte = n & 255      # 255 = 11111111₂: giữ 8 bit cuối
    result = (result << 8) | reverse_byte_table[byte]
    n >>= 8             # bỏ 8 bit cuối đã xử lý
```

Giải thích:
- `n & 255`: lấy 8 bit cuối.
- `reverse_byte_table[byte]`: tra đúng số có 8 bit đảo thứ tự.
- `result << 8`: chừa **8 vị trí** cho byte tiếp theo.
- `n >>= 8`: chuyển sang byte phía trái kế tiếp.

Chạy 4 lượt (tổng 32 bit) thay vì 32 lượt bit, nhưng mất thời gian/bộ nhớ tiền tính bảng. Tính O(1) theo chiều rộng 32 bit; nếu xét số lần gọi lớn, bảng có thể được tái sử dụng.

In [35]:
def reverse_bits_table(n):
    result=0
    for _ in range(4):
        byte=n&255
        result=(result<<8)|reverse_byte_table[byte]
        n>>=8
    return result

for n in [0,1,2,13,43261596,2147483644,(1<<32)-1]:
    assert reverse_bits_table(n)==reverse_bits_bitwise(n)
print('PASS tra bảng 256 byte: kết quả giống lời giải chính')

PASS tra bảng 256 byte: kết quả giống lời giải chính


## 41. Big-O của bảng tiền tính: đổi bộ nhớ để xử lý lặp nhanh hơn

Một lần tạo bảng `256` phần tử, mỗi phần tử tính đảo 8 bit. Với độ rộng cố định, đây là chi phí **hằng số một lần**; bảng cần **256 số nguyên** bộ nhớ.

- **Cách 32 bit**: mỗi lần gọi xử lý 32 bước bit.
- **Tra bảng byte**: mỗi lần gọi xử lý 4 bước byte, cộng thời gian tra cứu; có chi phí tạo bảng trước.

Không nên khẳng định bảng “nhanh hơn 8 lần” chỉ từ số lượt: Python có chi phí vòng lặp, tạo bảng, truy cập list và phép dịch, nên hiệu năng thực tế cần đo. Đây là **trade-off**: tăng bộ nhớ và tiền xử lý để giảm việc lặp lại khi xử lý nhiều input.

Với người mới, ưu tiên code 32 lượt dễ hiểu, đúng và dễ chứng minh. Biến thể tra bảng chỉ cần khi học **follow-up**.

## 42. Phân biệt các toán tử dễ nhìn nhầm

| Toán tử | Ý nghĩa | Với `n=13` |
|---|---|---|
| `n & 1` | AND theo bit, lấy bit cuối | `1` |
| `n >> 1` | Dịch phải, bỏ bit cuối | `6` |
| `n << 1` | Dịch trái, thêm bit 0 cuối | `26` |
| `n \| 1` | OR theo bit, bật bit cuối | `13` |
| `n ^ 1` | XOR, **đảo giá trị bit cuối** | `12` |
| `n // 2` | Chia nguyên 2 | `6` |

**Lưu ý:** `^` không phải phép lũy thừa trong Python; phép lũy thừa dùng `**`. Toán tử XOR là kiến thức thêm để bạn không nhầm dấu; lời giải chính #190 không cần XOR.

Bài này sử dụng `&`, `>>`, `<<`, `|`, nhưng **không** cần `^` hay `~`.

In [36]:
n=13
print('n      ',n,bin(n))
print('n & 1  ',n&1)
print('n >> 1 ',n>>1)
print('n << 1 ',n<<1)
print('n | 1  ',n|1)
print('n ^ 1  ',n^1)
print('2 ** 3 ',2**3)

n       13 0b1101
n & 1   1
n >> 1  6
n << 1  26
n | 1   13
n ^ 1   12
2 ** 3  8


# 43. TỰ LUYỆN — 6 bài (đừng đọc đáp án ở dưới ngay)

Lộ trình:
1. Dự đoán output Python (`&`, `<<`, `>>`, `|`).
2. Trace bằng tay phép đảo **8 bit**.
3. Điền TODO để tự viết lại thuật toán 32 bit.
4. Tự cài đặt cách giải bằng chuỗi 32 bit.
5. Biến thể đảo w bit theo tham số.
6. Mini challenge: kiểm thử tính chất đảo hai lần và so sánh hai lời giải.

Các cell TODO chỉ định nghĩa hàm/biến, **không tự gọi hàm chưa hoàn thành**. Muốn kiểm tra, bạn tự bật những dòng `assert` đang được comment. Gợi ý ở cuối toàn bộ bài tập, đáp án ở sau phần gợi ý.

### Bài 1 — Dự đoán output của những toán tử bit

Với `n = 13` (nhị phân `1101`), hãy tính tay:

1. `n & 1` = ?
2. `n >> 1` = ?
3. `n << 1` = ?
4. `(n << 1) | 1` = ?
5. `format(n, '08b')` trả chuỗi nào?

**Đừng chạy code cho tới khi tự ghi đủ 5 câu.** Hãy vẽ 4–8 ô bit để xác định vị trí.

In [37]:
exercise1_guess190 = None  # TODO: ghi lại 5 kết quả bằng tay
# Khi tự làm xong, có thể kiểm tra bằng:
# n=13
# print(n&1,n>>1,n<<1,(n<<1)|1,format(n,'08b'))

### Bài 2 — Trace thủ công đảo **8 bit** của số 6

`6` viết theo 8 bit là `00000110`.

1. Đảo 8 ký tự trên giấy được chuỗi gì? Số nguyên kết quả bao nhiêu?
2. Lập bảng **8 lượt**: `n trước`, `bit=n&1`, `result sau`, `n sau`.
3. Sau mấy lượt n trở thành 0? Vì sao các lượt còn lại vẫn cần thực hiện?

Nhớ: đây là bài **minh họa 8 bit**, không phải lời giải trực tiếp của #190 (32 bit).

In [38]:
exercise2_trace190=None  # TODO: điền kết quả n=6, width=8
# Sau khi tự làm, bạn có thể chạy:
# trace_reverse_bits(6,8)

### Bài 3 — TODO: viết lại hàm đảo 32 bit không dùng chuỗi

Hãy tự điền:

- Bắt đầu `result=0`.
- Vòng `for` đúng **32** lượt.
- Mỗi lượt lấy bit cuối của n, dịch result trái 1, OR bit vào, dịch n phải 1.
- Trả về `result`.

Nếu bạn không nhớ thứ tự `&` và `>>`, hãy tự chạy lại ví dụ một bit n=1 **bằng tay** trước.

In [39]:
def exercise3_reverse_bits190(n):
    # TODO 1: result = 0
    # TODO 2: for _ in range(32):
    # TODO 3: bit = n & 1
    # TODO 4: result = (result << 1) | bit
    # TODO 5: n >>= 1
    # TODO 6: return result
    pass

# Sau khi tự viết, bật:
# assert exercise3_reverse_bits190(43261596)==964176192

### Bài 4 — Tự viết cách chuỗi: `format`, `[::-1]`, `int(...,2)`

Hàm `exercise4_reverse_string190(n)` cần:

1. Chuyển n thành chuỗi 32 bit.
2. Đảo chuỗi.
3. Đổi chuỗi sau đảo thành số nguyên hệ 2.
4. `return` số nguyên.

**Thử n=2**, đây là ca dễ phát hiện lỗi không đệm đủ 32 bit.

In [40]:
def exercise4_reverse_string190(n):
    # TODO 1: bits = format(n,'032b')
    # TODO 2: reversed_bits = bits[::-1]
    # TODO 3: return int(reversed_bits,2)
    pass

# assert exercise4_reverse_string190(2)==1073741824

### Bài 5 — Biến thể: đảo `width` bit

Viết `exercise5_reverse_width190(n,width)`:

- Chỉ xử lý n trong khoảng `0 ≤ n < 2^width`.
- Chạy đúng width vòng với `n&1`, dịch/gắn kết quả và `n>>1`.
- Với 4 bit, `n=13` → 11.
- Với 8 bit, `n=13` → 176.

Có thể dùng `raise ValueError(...)` khi width không dương hoặc n không biểu diễn vừa width bit. Đây là bài mở rộng tổng quát hóa, không phải thay đổi format submit.

In [41]:
def exercise5_reverse_width190(n,width):
    # TODO 1: kiểm tra 0 <= n < 2**width và width>=1
    # TODO 2: đảo đúng width lượt bằng toán tử bit
    # TODO 3: return số nguyên
    pass

# assert exercise5_reverse_width190(13,4)==11
# assert exercise5_reverse_width190(13,8)==176

### Bài 6 — Mini challenge: property test và kiểm tra chéo

Viết `exercise6_check190(values)`:

- Duyệt tất cả n trong values.
- Tính `reverse_bits_bitwise(n)` và `reverse_bits_string(n)`; **so sánh hai kết quả**.
- Đảo kết quả bitwise thêm lần nữa, kiểm tra có bằng n không.
- Trả `True` khi mọi test đạt, `False` nếu có bất kỳ test sai.

Thử các số `0, 1, 2, 13, 43261596, 2³²−1`. Đây là bước luyện **kiểm chứng một property** và kiểm thử hai thuật toán độc lập.

In [42]:
def exercise6_check190(values):
    # TODO 1: duyệt từng n
    # TODO 2: nếu reverse_bits_bitwise(n) != reverse_bits_string(n): return False
    # TODO 3: nếu reverse_bits_bitwise(reverse_bits_bitwise(n)) != n: return False
    # TODO 4: return True
    pass

# assert exercise6_check190([0,1,2,13,43261596,(1<<32)-1]) is True

## 44. GỢI Ý 3 CẤP — hãy dùng từ cấp nhẹ nhất

| Bài | Cấp 1: suy nghĩ | Cấp 2: công cụ | Cấp 3: gần code |
|---|---|---|---|
| **1** | Viết 13 = `1101` | `&1` lấy bit cuối; `>>` bỏ bit | 1; 6; 26; 27; `'00001101'` |
| **2** | 6 có 8 bit `00000110` | Đảo chuỗi hoặc trace 8 lượt | `01100000` = 96; n=0 sau lượt 3 |
| **3** | Đọc từ phải, ghép sang phải | `bit=n&1`; `n>>=1` | `result=(result<<1)|bit` trong `range(32)` |
| **4** | Đầu tiên phải có **đủ 32 bit** | `format(n,'032b')` | `return int(bits[::-1],2)` |
| **5** | Giống bài chính, thay 32 bằng width | `0 <= n < (1<<width)` | `for _ in range(width)` |
| **6** | Đảo hai lần quay lại ban đầu | Dùng hai hàm có sẵn | `if bitwise(n)!=string(n): return False`; kiểm tra roundtrip |

**Bạn nên tự hoàn thiện TODO trước khi đọc lời giải dưới đây.**

# 45. ĐÁP ÁN THAM KHẢO — chỉ mở khi đã thử

## Đáp án bài 1 và bài 2 (giải tay)

**Bài 1:**

- `13 & 1 = 1` vì `1101` kết thúc bằng bit 1.
- `13 >> 1 = 6` vì `1101` bỏ bit cuối còn `110`.
- `13 << 1 = 26` vì `1101` thêm bit 0 thành `11010`.
- `(13 << 1) | 1 = 27` vì `11010 OR 00001 = 11011`.
- `format(13, '08b') = '00001101'`.

**Bài 2:** `6 = 00000110₂` → đảo 8 bit thành `01100000₂` = `64+32=96`. Lần 1 lấy bit 0, lần 2 lấy bit 1, lần 3 lấy bit 1, n trở thành 0 sau lần 3. 5 lần còn lại vẫn dịch result sang trái để đủ 8 bit.

In [43]:
assert (13&1,13>>1,13<<1,(13<<1)|1,format(13,'08b'))==(1,6,26,27,'00001101')
assert reverse_bits_width(6,8)==96
print('PASS đáp án bài 1 và bài 2')

PASS đáp án bài 1 và bài 2


### Đáp án bài 3 — bitwise 32 lượt

Ta phải lấy bit cuối **trước khi** dịch n. `result` dịch trái trước khi OR với bit, vì cần một chỗ trống ở bên phải. Vòng lặp đúng 32 lần đảm bảo các bit 0 phía trái cũng được xử lý.

In [44]:
def answer3_reverse_bits190(n):
    result=0
    for _ in range(32):
        bit=n&1
        result=(result<<1)|bit
        n>>=1
    return result

assert answer3_reverse_bits190(43261596)==964176192
assert answer3_reverse_bits190(2)==1073741824
print('PASS bài 3')

PASS bài 3


### Đáp án bài 4 — giải bằng chuỗi đủ 32 bit

Dùng `format(n,'032b')` thay vì `bin(n)` vì ta **cần cả những số 0 dẫn đầu**. Đảo `bits[::-1]` chỉ cho ra chuỗi; `int(...,2)` mới chuyển sang số nguyên là dạng `return` đề yêu cầu.

In [45]:
def answer4_reverse_string190(n):
    bits=format(n,'032b')
    reversed_bits=bits[::-1]
    return int(reversed_bits,2)

assert answer4_reverse_string190(2)==1073741824
assert answer4_reverse_string190(43261596)==964176192
print('PASS bài 4')

PASS bài 4


### Đáp án bài 5 — đảo w bit có xác nhận đầu vào

`width` là số lượt đọc bit. Với `width=4`, số 13 vừa 4 bit; với `width=8`, ta phải đọc thêm bốn bit 0 dẫn đầu. Kiểm tra đầu vào giúp tránh một số có 9 bit bị **truncated** khi đảo chỉ 8 bit mà không báo lỗi.

In [46]:
def answer5_reverse_width190(n,width):
    if width<1 or not(0<=n<(1<<width)):
        raise ValueError('n không nằm trong độ rộng đã cho')
    result=0
    for _ in range(width):
        result=(result<<1)|(n&1)
        n>>=1
    return result

assert answer5_reverse_width190(13,4)==11
assert answer5_reverse_width190(13,8)==176
assert answer5_reverse_width190(13,32)==reverse_bits_bitwise(13)
print('PASS bài 5')

PASS bài 5


### Đáp án bài 6 — kiểm thử bằng hai tính chất

Nếu có một giá trị n mà một trong hai tính chất sai, ta trả về `False` **ngay khi phát hiện**:

- Hai lời giải string/bitwise phải trùng nhau.
- Đảo bitwise hai lần phải trở về n.

Nếu chạy hết mọi n mà không phát hiện lỗi, trả `True`. Việc kiểm thử hai tính chất là một kỹ thuật hữu ích để tự kiểm chứng thuật toán trên dữ liệu đa dạng.

In [47]:
def answer6_check190(values):
    for n in values:
        reversed_value=reverse_bits_bitwise(n)
        if reversed_value!=reverse_bits_string(n):
            return False
        if reverse_bits_bitwise(reversed_value)!=n:
            return False
    return True

assert answer6_check190([0,1,2,13,43261596,(1<<32)-1]) is True
print('PASS bài 6')

PASS bài 6


## 46. Template tự suy nghĩ và checklist nộp bài

**Khi gặp bài đảo bit có độ rộng cố định:**

1. Độ rộng của dãy bit là bao nhiêu? Ở #190: **32**.
2. Ta đọc bit từ phía nào? Muốn đảo thứ tự, hãy lấy **bit cuối của n** trước.
3. Kết quả cần lưu ở đâu? `result` là số chứa các bit đã ghép.
4. Ghép bit mới thế nào? `result = (result << 1) | (n & 1)`.
5. Chuyển sang bit kế tiếp? `n >>= 1`.
6. Dừng khi nào? **Sau đúng 32 lượt**, không phải khi n=0.

**Code nộp gọn:**

```python
class Solution:
    def reverseBits(self, n: int) -> int:
        result = 0
        for _ in range(32):
            result = (result << 1) | (n & 1)
            n >>= 1
        return result
```

**Checklist cuối:** [ ] đảo bit không đảo số thập phân [ ] đủ 32 bit [ ] lấy bit trước khi dịch n [ ] OR chứ không AND để ghép [ ] `return` số nguyên [ ] đúng `class Solution` và `reverseBits` [ ] pass các ví dụ và test biên.

**Bạn đã học được:** thao tác bit từ cơ bản, cách suy ra thuật toán thay vì học thuộc, cách trace trạng thái và cách tự kiểm thử một lời giải. Các kỹ thuật này hữu ích cho các bài như `Number of 1 Bits`, `Counting Bits`, `Single Number` và nhiều bài bitwise khác.